In [ ]:
!pip install -q google-genai

In [ ]:
from google.colab import userdata
import os

os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')

In [ ]:
from google import genai

cliente = genai.Client()

In [ ]:
respuesta = cliente.models.generate_content(
    model="gemini-2.5-flash",
    contents="Cuál es la capital y la ciudad más grande de Turquía?"
)

print(respuesta.text)

La capital de Turquía es **Ankara**.

La ciudad más grande de Turquía es **Estambul**.


In [ ]:
from google.colab import files

os.makedirs("PDFs", exist_ok=True)
uploader = files.upload()

for archivo in uploader.keys():
  os.rename(archivo, f"PDFs/{archivo}")

Saving Carrarurquia_Reporte_Q1_2025.pdf to Carrarurquia_Reporte_Q1_2025.pdf
Saving Carrarurquia_Reporte_Q2_2025.pdf to Carrarurquia_Reporte_Q2_2025.pdf
Saving Carrarurquia_Reporte_Q3_2025.pdf to Carrarurquia_Reporte_Q3_2025.pdf
Saving Carrarurquia_Reporte_Q4_2025.pdf to Carrarurquia_Reporte_Q4_2025.pdf


In [ ]:
!pip install -q langchain-community pypdf

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

In [ ]:
documentos = []

for archivo2 in os.listdir("PDFs"):
  ruta = os.path.join("PDFs", archivo2)
  loader = PyPDFLoader(ruta)
  paginas = loader.load()
  documentos.extend(paginas)

In [ ]:
documentos[0]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:49:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:49:44+00:00', 'subject': '(unspecified)', 'title': 'Carrarurquía - Reporte Q2 2025', 'trapped': '/False', 'source': 'PDFs/Carrarurquia_Reporte_Q2_2025.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}, page_content='Carrarurquía\nReporte trimestral Q2 2025 (ficticio)\nPeriodo: 01/04/2025 - 30/06/2025\nMoneda: USD\n1\n Carrarurquía\n Reporte trimestral Q2 2025 · Viajes a Turquía (ficticio)\n Periodo: 01/04/2025 - 30/06/2025\n Moneda de referencia: dólares estadounidenses (USD)\nEste documento presenta resultados y aprendizajes ficticios del trimestre para Carrarurquía, una\nempresa de viajes especializada en experiencias en Turquía. Los datos, cifras y testimonios han sido\ncreados con fines demostrativos y no representan operaciones reales.\n Edición: Julio de 2025')

In [ ]:
len(documentos)

60

In [ ]:
!pip install -q langchain-text-splitters

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

divisor = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=40,
    separators=["\n\n", "\n", ". ", " ", ""]
)

fragmentos = divisor.split_documents(documentos)

In [ ]:
fragmentos[67]

Document(metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:30:27+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:30:27+00:00', 'subject': '(unspecified)', 'title': '(anonymous)', 'trapped': '/False', 'source': 'PDFs/Carrarurquia_Reporte_Q1_2025.pdf', 'total_pages': 15, 'page': 8, 'page_label': '9'}, page_content='5. Operaciones y calidad\nEntrega del viaje\nEn Q1 se operaron 68 salidas (grupales y semi-privadas), con un cumplimiento de\nitinerario del 93%. Las incidencias se concentraron en traslados interurbanos en días de\nlluvia y en reprogramaciones por disponibilidad de actividades.\nCadena de suministro y proveedores\nProveedor\nTipo\nRegión\nParticipación de coste\nEvaluación (1-5)\nBosporus DMC\nOperador local')

In [ ]:
!pip install -q langchain-google-genai faiss-cpu

In [ ]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
len(fragmentos)

172

In [ ]:
vectorstore1 = FAISS.from_documents(
    documents=fragmentos[0:89],
    embedding=embeddings
)

In [ ]:
vectorstore2 = FAISS.from_documents(
    documents=fragmentos[90:],
    embedding=embeddings
)

In [ ]:
vectorstore1.index.reconstruct(0)

array([-0.00143831,  0.00889298,  0.01531782, ...,  0.01553657,
       -0.00974552, -0.02190461], dtype=float32)

In [ ]:
len(vectorstore1.index.reconstruct(0))

3072

In [ ]:
vectorstore1.merge_from(vectorstore2)

In [ ]:
consulta = "Cuál es el paquete de viajes más económico de Carrarurquía?"

resultados = vectorstore1.similarity_search(
    consulta,
    k=7
)

for i in resultados:
  print(i)
  print("\n")

page_content='Carrarurquía
Reporte trimestral Q3 2025 (ficticio)
Periodo: 01/07/2025 - 30/09/2025
Moneda: USD
13
11. Apéndice A: Paquetes y tarifas de referencia
Lista simplificada de precios (USD)
Tarifas ficticias orientativas para paquetes populares. Los importes se muestran por persona en ocupación' metadata={'producer': 'ReportLab PDF Library - www.reportlab.com', 'creator': '(unspecified)', 'creationdate': '2026-02-13T13:49:44+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-02-13T13:49:44+00:00', 'subject': '(unspecified)', 'title': 'Carrarurquía - Reporte Q3 2025', 'trapped': '/False', 'source': 'PDFs/Carrarurquia_Reporte_Q3_2025.pdf', 'total_pages': 15, 'page': 12, 'page_label': '13'}


page_content='Carrarurquía
Reporte trimestral Q4 2025 (ficticio)
Periodo: 01/10/2025 - 31/12/2025
Moneda: USD
13
11. Apéndice A: Paquetes y tarifas de referencia
Lista simplificada de precios (USD)
Tarifas ficticias orientativas para paquetes populares. Los importes se muestran

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0.2
)

retriever = vectorstore1.as_retriever(
    search_kwargs={"k": 4}
)

In [ ]:
def preguntar_rag(pregunta):
    """Busca contexto relevante en los documentos y genera una respuesta."""
    # Paso 1: Buscar los chunks más relevantes
    docs = retriever.invoke(pregunta)
    contexto = "\n\n---\n\n".join(doc.page_content for doc in docs)

    # Paso 2: Construir el prompt con el contexto encontrado
    prompt = f"""Eres un asistente experto que responde preguntas basándose ÚNICAMENTE
    en el contexto proporcionado. Si la información no está en el contexto,
    di que no tienes suficiente información.

    Contexto: {contexto}

    Pregunta: {pregunta}

    Respuesta:"""

    # Paso 3: Enviar al modelo y devolver la respuesta
    respuesta = llm.invoke(prompt)
    return respuesta.content

In [ ]:
preguntar_rag("Dónde se mantuvo concentrado el mix de productos?")

'El mix de producto se mantuvo concentrado en circuitos combinados (Estambul + Capadocia).'

In [ ]:
preguntar_rag("Cuántos mundiales de fútbol tiene Brasil?")

'No tengo suficiente información en el contexto proporcionado para responder cuántos mundiales de fútbol tiene Brasil.'